In [0]:
%sql
CREATE DATABASE IF NOT EXISTS delta_training;

USE delta_training;

In [0]:
customers_data = [
    (1, "Aarav Sharma", "Hyderabad", "Gold", 25000),
    (2, "Priya Reddy", "Bengaluru", "Silver", 18000),
    (3, "Rohan Mehta", "Mumbai", "Gold", 32000),
    (4, "Sneha Iyer", "Chennai", "Bronze", 12000),
    (5, "Kiran Patel", "Hyderabad", "Silver", 22000),
    (6, "Ananya Das", "Kolkata", "Gold", 41000),
    (7, "Vikram Singh", "Delhi", "Bronze", 9000),
    (8, "Meera Nair", "Bengaluru", "Silver", 26000)
]

columns = [
    "customer_id",
    "customer_name",
    "city",
    "membership",
    "total_spend"
]

customers_df = spark.createDataFrame(customers_data, columns)

display(customers_df)

customer_id,customer_name,city,membership,total_spend
1,Aarav Sharma,Hyderabad,Gold,25000
2,Priya Reddy,Bengaluru,Silver,18000
3,Rohan Mehta,Mumbai,Gold,32000
4,Sneha Iyer,Chennai,Bronze,12000
5,Kiran Patel,Hyderabad,Silver,22000
6,Ananya Das,Kolkata,Gold,41000
7,Vikram Singh,Delhi,Bronze,9000
8,Meera Nair,Bengaluru,Silver,26000


In [0]:
%sql
CREATE OR REPLACE TABLE customer_delta(
  customer_id INT,
  customer_name STRING,
  city STRING,
  membership STRING,
  total_spend INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO customer_delta VALUES
(1, 'John', 'NY', 'Gold', 1000),
(2, 'Jane', 'LA', 'Silver', 500),
(3, 'Bob', 'NY', 'Gold', 1500),
(4, 'Alice', 'LA', 'Bronze', 200);

SELECT * FROM customer_delta;

customer_id,customer_name,city,membership,total_spend
1,John,NY,Gold,1000
2,Jane,LA,Silver,500
3,Bob,NY,Gold,1500
4,Alice,LA,Bronze,200


In [0]:
from pyspark.sql.functions import col

customers_df = customers_df.withColumn(
    "customer_id",
    col("customer_id").cast("int")
).withColumn(
    "total_spend",
    col("total_spend").cast("int")
)

In [0]:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_delta")

In [0]:
customers_df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- membership: string (nullable = true)
 |-- total_spend: long (nullable = true)



In [0]:
delta_path = "/tmp/delta/employees"

df.write.format("delta") \
    .mode("overwrite") \
    .save(delta_path)

In [0]:
path_df = spark.read.format("delta").load(delta_path)
display(path_df)

In [0]:
%sql
CREATE OR REPLACE TABLE retail_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA

In [0]:
%sql
INSERT INTO retail_orders VALUES
(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),
(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),
(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),
(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed');

num_affected_rows,num_inserted_rows
6,6


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW new_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed')
AS new_orders(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
%sql
MERGE INTO retail_orders AS target
USING new_orders AS source
ON target.order_id = source.order_id

 

WHEN MATCHED THEN
UPDATE SET
target.customer_name = source.customer_name,
target.city = source.city,
target.product = source.product,
target.quantity = source.quantity,
target.price = source.price,
target.order_status = source.order_status

 

WHEN NOT MATCHED THEN
INSERT (
order_id,
customer_name,
city,
product,
quantity,
price,
order_status
)

 

VALUES (
source.order_id,
source.customer_name,
source.city,
source.product,
source.quantity,
source.price,
source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql

INSERT INTO retail_orders VALUES
(101, 'Viswa', 'Chennai', 'Laptop', 2, 75000, 'Confirmed'),
(102, 'Arun', 'Bengaluru', 'Mobile', 1, 25000, 'Pending');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql

UPDATE retail_orders
SET price = 78000
WHERE product = 'Laptop';

num_affected_rows
3


In [0]:
%sql

DELETE FROM retail_orders
WHERE quantity = 1;

num_affected_rows
5


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW incoming_orders AS

SELECT 101 AS order_id,
       'Viswa' AS customer_name,
       'Chennai' AS city,
       'Laptop' AS product,
       3 AS quantity,
       78000 AS price,
       'Delivered' AS order_status

UNION ALL

SELECT 103,
       'Priya',
       'Hyderabad',
       'Tablet',
       2,
       30000,
       'Confirmed';

In [0]:
%sql

MERGE INTO retail_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
target.customer_name = source.customer_name,
target.city = source.city,
target.product = source.product,
target.quantity = source.quantity,
target.price = source.price,
target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT (
order_id,
customer_name,
city,
product,
quantity,
price,
order_status
)

VALUES (
source.order_id,
source.customer_name,
source.city,
source.product,
source.quantity,
source.price,
source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


# **ASSIGNMENT_!**


In [0]:
%sql CREATE OR REPLACE TABLE ecommerce_orders ( order_id INT, customer_name STRING, city STRING, product STRING, quantity INT, price INT, order_status STRING ) USING DELTA; 

In [0]:
%sql INSERT INTO ecommerce_orders VALUES (101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'), (102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'), (103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'), (104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'), (105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'), (106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed'), (107,'Vikram Singh','Delhi','Camera',1,55000,'Placed'), (108,'Meera Nair','Bangalore','Laptop',1,80000,'Placed'); 

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql

INSERT INTO ecommerce_orders
VALUES
(109,'Rahul Verma','Mumbai','Tablet',1,27000,'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql

INSERT INTO ecommerce_orders
VALUES
(110,'Arjun Nair','Kochi','Mobile',1,28000,'Placed'),
(111,'Kavya Reddy','Hyderabad','Laptop',1,79000,'Placed');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql

INSERT INTO ecommerce_orders
VALUES
(112,'Farhan Ali','Delhi','Camera',1,45000,'Shipped');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql

INSERT INTO ecommerce_orders
VALUES
(113,'Suresh Kumar','Chennai','Headphones',5,2500,'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql

INSERT INTO ecommerce_orders
VALUES
(114,'Anil Reddy','Hyderabad','Mobile',1,26000,'Placed'),
(115,'Deepika Rao','Hyderabad','Laptop',1,82000,'Placed'),
(116,'Ramesh Kumar','Hyderabad','Tablet',2,24000,'Placed');

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql

UPDATE ecommerce_orders
SET order_status = 'Shipped'
WHERE order_id = 101;

num_affected_rows
1


In [0]:
%sql

UPDATE ecommerce_orders
SET quantity = quantity + 1
WHERE order_id = 102;

num_affected_rows
1


In [0]:

%sql

UPDATE ecommerce_orders
SET price = 78000
WHERE product = 'Laptop';

num_affected_rows
5


In [0]:
%sql

UPDATE ecommerce_orders
SET city = 'Secunderabad'
WHERE customer_name = 'Amit Sharma';

num_affected_rows
1


In [0]:
%sql

UPDATE ecommerce_orders
SET order_status = 'Delivered'
WHERE product = 'Mobile';

num_affected_rows
4


In [0]:
%sql

UPDATE ecommerce_orders
SET price = 2500
WHERE product = 'Headphones';

num_affected_rows
2


In [0]:
%sql

UPDATE ecommerce_orders
SET price = price + 1000
WHERE product = 'Tablet';

num_affected_rows
3


In [0]:
%sql

UPDATE ecommerce_orders
SET order_status = 'Processing'
WHERE city = 'Bangalore';

num_affected_rows
2


In [0]:
%sql

UPDATE ecommerce_orders
SET quantity = 2
WHERE quantity = 1;

num_affected_rows
11


In [0]:
%sql

UPDATE ecommerce_orders
SET city = 'Surat'
WHERE city = 'Ahmedabad';

num_affected_rows
1


In [0]:
%sql

DELETE FROM ecommerce_orders
WHERE order_status = 'Cancelled';

num_affected_rows
1


In [0]:
%sql

DELETE FROM ecommerce_orders
WHERE quantity > 3;

num_affected_rows
1


In [0]:
%sql

DELETE FROM ecommerce_orders
WHERE product = 'Camera';

num_affected_rows
2


In [0]:
%sql

DELETE FROM ecommerce_orders
WHERE city = 'Kolkata';

num_affected_rows
1


In [0]:
%sql

DELETE FROM ecommerce_orders
WHERE price < 5000;

num_affected_rows
1


In [0]:
%sql

DELETE FROM ecommerce_orders
WHERE customer_name LIKE 'A%';

num_affected_rows
3


In [0]:
%sql

DELETE FROM ecommerce_orders
WHERE product = 'Tablet';

num_affected_rows
2


In [0]:
%sql

DELETE FROM ecommerce_orders
WHERE city = 'Mumbai'
AND quantity = 1;

num_affected_rows
0


In [0]:
%sql

DELETE FROM ecommerce_orders
WHERE price > 80000;

num_affected_rows
0


In [0]:
%sql

DELETE FROM ecommerce_orders
WHERE order_id = (
    SELECT MAX(order_id)
    FROM ecommerce_orders
);

num_affected_rows
1


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW incoming_orders AS

SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed'),
(111,'Farhan Ali','Hyderabad','Laptop',1,79000,'Placed')

AS incoming_orders(
order_id,
customer_name,
city,
product,
quantity,
price,
order_status
);

In [0]:
%sql

MERGE INTO ecommerce_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
target.customer_name = source.customer_name,
target.city = source.city,
target.product = source.product,
target.quantity = source.quantity,
target.price = source.price,
target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT (
order_id,
customer_name,
city,
product,
quantity,
price,
order_status
)

VALUES (
source.order_id,
source.customer_name,
source.city,
source.product,
source.quantity,
source.price,
source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,3,0,2


In [0]:
%sql

SELECT *
FROM ecommerce_orders
ORDER BY order_id;

order_id,customer_name,city,product,quantity,price,order_status
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
110,Divya Menon,Kochi,Mobile,1,27000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed


In [0]:
%sql

SELECT
source.order_id,

CASE
WHEN target.order_id IS NOT NULL THEN 'UPDATED'
ELSE 'INSERTED'
END AS action

FROM incoming_orders source

LEFT JOIN ecommerce_orders target
ON source.order_id = target.order_id;

order_id,action
102,UPDATED
104,UPDATED
109,UPDATED
110,UPDATED
111,UPDATED


In [0]:
%sql

MERGE INTO ecommerce_orders target
USING incoming_orders source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
target.order_status = source.order_status;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
%sql

MERGE INTO ecommerce_orders target
USING incoming_orders source
ON target.order_id = source.order_id

WHEN MATCHED
AND source.quantity > target.quantity

THEN UPDATE SET
target.quantity = source.quantity;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql

MERGE INTO ecommerce_orders target
USING incoming_orders source
ON target.order_id = source.order_id

WHEN MATCHED
AND source.order_status != 'Cancelled'

THEN UPDATE SET
target.order_status = source.order_status;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
%sql

MERGE INTO ecommerce_orders target
USING incoming_orders source
ON target.order_id = source.order_id

WHEN NOT MATCHED
AND source.product = 'Laptop'

THEN INSERT (
order_id,
customer_name,
city,
product,
quantity,
price,
order_status
)

VALUES (
source.order_id,
source.customer_name,
source.city,
source.product,
source.quantity,
source.price,
source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql

MERGE INTO ecommerce_orders target
USING incoming_orders source
ON target.order_id = source.order_id

WHEN MATCHED
AND source.price > target.price

THEN UPDATE SET
target.price = source.price;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW daily_updates AS

SELECT * FROM VALUES
(103,'Rohit Mehta','Mumbai','Headphones',5,2200,'Delivered'),
(106,'Ananya Das','Kolkata','Mobile',2,28000,'Shipped'),
(112,'Sanjay Gupta','Delhi','Tablet',1,24000,'Placed')

AS daily_updates(
order_id,
customer_name,
city,
product,
quantity,
price,
order_status
);

In [0]:
%sql

MERGE INTO ecommerce_orders target
USING daily_updates source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
target.customer_name = source.customer_name,
target.city = source.city,
target.product = source.product,
target.quantity = source.quantity,
target.price = source.price,
target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT (
order_id,
customer_name,
city,
product,
quantity,
price,
order_status
)

VALUES (
source.order_id,
source.customer_name,
source.city,
source.product,
source.quantity,
source.price,
source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,0,0,3


In [0]:
%sql

SELECT COUNT(*) AS updated_rows
FROM ecommerce_orders
WHERE order_id IN (103,106);

updated_rows
2


In [0]:
%sql

SELECT COUNT(*) AS inserted_rows
FROM ecommerce_orders
WHERE order_id = 112;

inserted_rows
1


In [0]:
%sql

SELECT *
FROM ecommerce_orders
ORDER BY price DESC;

order_id,customer_name,city,product,quantity,price,order_status
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
106,Ananya Das,Kolkata,Mobile,2,28000,Shipped
110,Divya Menon,Kochi,Mobile,1,27000,Placed
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
112,Sanjay Gupta,Delhi,Tablet,1,24000,Placed
103,Rohit Mehta,Mumbai,Headphones,5,2200,Delivered
